# 🎬 AI Movie Translate & Dubbing (v2.2) — Web UI Launcher

Google Colab ပေါ်တွင် **NVIDIA T4 GPU** စွမ်းအားဖြင့် **Web UI Dashboard** ကို 1-Click ဖွင့်လှစ်ပေးမည့် စနစ် ဖြစ်ပါသည်။

### 📌 စတင်ရန် လမ်းညွှန်ချက်:
1. မီးနူးမှ **Runtime > Change runtime type** သို့သွား၍ **T4 GPU** ရွေးချယ်ထားကြောင်း သေချာပါစေ။
2. အောက်ပါ Cell ၏ **Play (▶️)** ခလုတ်ကို နှိပ်လိုက်ရုံဖြင့် လိုအပ်သည်များကို အလိုအလျောက် သွင်းယူပြီး **Web UI Dashboard** ကို ချက်ချင်း ဖွင့်လှစ်ပေးပါမည်။
3. ဗီဒီယို Link ထည့်ခြင်း၊ API Key ထည့်ခြင်း၊ အသံရွေးချယ်ခြင်းနှင့် အခြား Setting များ အားလုံးကို **Web UI ပေါ်မှပင် အပြည့်အစုံ ဆက်လက် လုပ်ဆောင်နိုင်ပါသည်**။

In [ ]:
# @title 🚀 Launch Web UI Dashboard (1-Click)
import os, sys, subprocess, time, re, shutil, socket
from IPython.display import display, HTML, Javascript

# 1. Background Keep-Alive Heartbeat (Colab လိုင်းမပြတ်စေရန် ကာကွယ်ခြင်း)
try:
    display(Javascript('''
    setInterval(function(){
        try {
            var btn = document.querySelector("colab-connect-button");
            if (btn && btn.shadowRoot) {
                var connectBtn = btn.shadowRoot.querySelector("#connect");
                if (connectBtn) connectBtn.click();
            }
        } catch(e){}
    }, 60000);
    '''))
    print("⚡ Colab Auto-KeepAlive Active (Prevents idle disconnect)")
except Exception:
    pass

project_dir = "/content/ai-translate-agent"

# 2. Setup Repository (အသစ်ဆုံး Version သို့ အလိုအလျောက် Update လုပ်ခြင်း)
if not os.path.exists(project_dir):
    print("[*] 1/4 Cloning repository...")
    !git clone https://github.com/paipai1999/ai-translate-agent.git {project_dir}
else:
    print("[*] 1/4 Updating repository to latest commit...")
    !cd {project_dir} && git fetch origin main && git reset --hard origin/main

os.chdir(project_dir)
%cd /content/ai-translate-agent

if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

# Create config.json from config.example.json if missing
cfg_path = os.path.join(project_dir, "config.json")
cfg_example = os.path.join(project_dir, "config.example.json")
if not os.path.exists(cfg_path) and os.path.exists(cfg_example):
    shutil.copy(cfg_example, cfg_path)

os.makedirs("temp", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("movies", exist_ok=True)

# 3. Install System Dependencies & Myanmar Padauk Fonts
print("[*] 2/4 Checking & Installing system dependencies...")
if not os.path.exists("/usr/share/fonts/truetype/padauk/Padauk.ttf"):
    !apt-get update -qq && apt-get install -y -qq ffmpeg fonts-sil-padauk fonts-noto-cjk fonts-noto-core > /dev/null 2>&1
    !fc-cache -f > /dev/null 2>&1

# Ensure yt-dlp and requirements are up-to-date
!pip install -q -U yt-dlp
!pip install -q -r requirements.txt

# Install Cloudflared Tunnel
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# Stop any existing server processes
!pkill -f "web_ui.py" || true
!pkill -f "cloudflared" || true

# 4. Start Web UI Server
print("[*] 3/4 Starting Web UI Server...")
server_proc = subprocess.Popen(
    [sys.executable, "web_ui.py", "--port", "5000"],
    cwd=project_dir,
    stdout=open("/content/web_ui.log", "w"),
    stderr=subprocess.STDOUT
)

# Wait until Web UI port 5000 is actively accepting connections
server_ready = False
for _ in range(25):
    if server_proc.poll() is not None:
        break
    try:
        with socket.create_connection(("127.0.0.1", 5000), timeout=1):
            server_ready = True
            break
    except OSError:
        time.sleep(1)

if not server_ready:
    print("❌ Web UI failed to start! Crash log details:")
    with open("/content/web_ui.log", "r") as f:
        print(f.read())
else:
    print("[*] 4/4 Connecting Cloudflare Secure Tunnel...")
    tunnel_proc = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    
    tunnel_url = None
    start_t = time.time()
    while time.time() - start_t < 40:
        line = tunnel_proc.stdout.readline()
        if not line and tunnel_proc.poll() is not None:
            break
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
            
    if tunnel_url:
        display(HTML(f"""
        <div style="background: linear-gradient(135deg, #0d1117, #161b22); border: 2px solid #58a6ff; border-radius: 14px; padding: 28px; text-align: center; margin: 24px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; box-shadow: 0 8px 24px rgba(0,0,0,0.5);">
            <div style="font-size: 40px; margin-bottom: 8px;">🎬</div>
            <h2 style="color: #58a6ff; margin: 0 0 10px 0; font-size: 24px;">Web UI Dashboard အဆင်သင့် ဖြစ်ပါပြီ!</h2>
            <p style="color: #c9d1d9; font-size: 15px; margin: 0 0 20px 0;">အောက်ပါ ခလုတ်ကြီးကို နှိပ်၍ Browser Tab အသစ်ဖြင့် Web UI ကို တိုက်ရိုက် ဖွင့်ပါ 👇</p>
            <a href="{tunnel_url}" target="_blank" style="background: linear-gradient(135deg, #1f6feb, #238636); color: #ffffff; font-weight: bold; font-size: 19px; padding: 15px 36px; border-radius: 10px; text-decoration: none; display: inline-block; box-shadow: 0 4px 16px rgba(31, 111, 235, 0.4); letter-spacing: 0.5px;">
                🌐 Open Web UI Dashboard ↗️
            </a>
            <div style="margin-top: 18px;">
                <span style="color: #8b949e; font-size: 13px;">Direct Link: </span>
                <a href="{tunnel_url}" target="_blank" style="color: #58a6ff; font-weight: 500; font-size: 14px; text-decoration: underline;">{tunnel_url}</a>
            </div>
        </div>
        """))
        print(f"\n👉 Direct URL: {tunnel_url}")
        try:
            display(Javascript(f'window.open("{tunnel_url}", "_blank");'))
        except Exception:
            pass
        tunnel_proc.wait()
    else:
        print("❌ Could not obtain Cloudflare tunnel URL. Web UI logs:")
        with open("/content/web_ui.log", "r") as f:
            print(f.read())
